# RBFE Cinnabar Analysis Example

This notebook analyzes a BATTER RBFE Cinnabar bundle, such as
`results/cinnabar/<run_id>/` produced after an RBFE run or a combined
bundle produced with `batter fe cinnabar`. It loads the stable BATTER CSVs,
optionally joins experimental affinities, compares calculated and
experimental DDGs/DGs, and displays the generated Cinnabar assets.

Set `CINNABAR_DIR` and, if available, `EXPERIMENTAL_DATA` in the configuration
cell before running the notebook.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    sns = None

try:
    from IPython.display import Image, display, HTML
except ImportError:
    def display(obj=None, *args, **kwargs):
        if obj is not None:
            print(obj)

    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename

        def __repr__(self):
            return f"Image(filename={self.filename!r})"

    class HTML(str):
        pass

from batter.api import read_cinnabar_outputs
from batter.analysis.cinnabar import dataframe_to_cinnabar

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
if sns is not None:
    sns.set_theme(style="whitegrid", context="notebook")
else:
    plt.style.use("default")


## Configure Inputs

`CINNABAR_DIR` should point at a directory containing files such as
`cinnabar_relative.csv`, `cinnabar_absolute.csv`, `edge_summary.csv`,
`raw_signed.csv`, and `manifest.json`.

`EXPERIMENTAL_DATA` is optional. It can be a CSV/TSV file with ligand IDs and
absolute binding free energies, or an SDF with an `r_exp_dg` property. When it
is missing or set to `None`, the notebook still summarizes the computed RBFE
network and skips experimental comparison plots.


In [ ]:
CINNABAR_DIR = Path("tyk2_rbfe/results/cinnabar/rep3")
EXPERIMENTAL_DATA = Path("tyk2_ligands.sdf")  # CSV/TSV or SDF; set to None to skip
OUTPUT_DIR = Path("rbfe_cinnabar_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_CYCLE_CLOSURE = True
EXCLUDED_LIGANDS = []
# Example for charge-annihilation benchmarks where alternate forms should be skipped:
# EXCLUDED_LIGANDS = ["EJM_52CHARG", "JMC_32CHARG"]

required_files = [
    "cinnabar_relative.csv",
    "edge_summary.csv",
    "raw_signed.csv",
    "manifest.json",
]
missing = [name for name in required_files if not (CINNABAR_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing expected Cinnabar files in {CINNABAR_DIR}: {missing}")

print(f"Cinnabar bundle: {CINNABAR_DIR.resolve()}")
print(f"Analysis output: {OUTPUT_DIR.resolve()}")


## Load BATTER Cinnabar Outputs


In [ ]:
relative_df, absolute_df = read_cinnabar_outputs(CINNABAR_DIR, require_absolute=False)

raw_signed = pd.read_csv(CINNABAR_DIR / "raw_signed.csv")
edge_summary = pd.read_csv(CINNABAR_DIR / "edge_summary.csv")
manifest = json.loads((CINNABAR_DIR / "manifest.json").read_text())

cycle_closure_edges_path = CINNABAR_DIR / "cycle_closure_edges.csv"
cycle_closure_nodes_path = CINNABAR_DIR / "cycle_closure_nodes.csv"
cycle_closure_edges = (
    pd.read_csv(cycle_closure_edges_path) if cycle_closure_edges_path.exists() else pd.DataFrame()
)
cycle_closure_nodes = (
    pd.read_csv(cycle_closure_nodes_path) if cycle_closure_nodes_path.exists() else pd.DataFrame()
)

summary = pd.DataFrame([
    {"table": "relative_df", "rows": len(relative_df), "columns": len(relative_df.columns)},
    {"table": "absolute_df", "rows": len(absolute_df), "columns": len(absolute_df.columns)},
    {"table": "raw_signed", "rows": len(raw_signed), "columns": len(raw_signed.columns)},
    {"table": "edge_summary", "rows": len(edge_summary), "columns": len(edge_summary.columns)},
    {"table": "cycle_closure_edges", "rows": len(cycle_closure_edges), "columns": len(cycle_closure_edges.columns)},
    {"table": "cycle_closure_nodes", "rows": len(cycle_closure_nodes), "columns": len(cycle_closure_nodes.columns)},
])
display(summary)
manifest


In [ ]:
display(relative_df)
if not absolute_df.empty:
    display(absolute_df)


## Load Experimental Data


In [ ]:
def normalize_ligand_id(label):
    return str(label).replace("-", "_").upper()


def cinnabar_label(label):
    return str(label).replace("-", "_").lower()


def load_experimental_sdf(path):
    """Load ligand IDs and experimental DG values from an SDF.

    RDKit is used when available. The fallback parser is enough for SDFs
    with a ligand name on the first non-empty line and an `r_exp_dg` property.
    """
    path = Path(path)
    try:
        from rdkit import Chem
    except ImportError:
        Chem = None

    rows = []
    if Chem is not None:
        supplier = Chem.SDMolSupplier(str(path))
        for mol in supplier:
            if mol is None:
                continue
            name = mol.GetProp("_Name")
            exp_dg = float(mol.GetProp("r_exp_dg"))
            rows.append({"ligand_id": name, "exp_dg": exp_dg})
    else:
        text = path.read_text()
        for record in text.split("$$$$"):
            lines = [line.strip() for line in record.splitlines() if line.strip()]
            if not lines:
                continue
            name = lines[0]
            exp_dg = np.nan
            for idx, line in enumerate(lines):
                if line == "> <r_exp_dg>" and idx + 1 < len(lines):
                    exp_dg = float(lines[idx + 1])
                    break
            rows.append({"ligand_id": name, "exp_dg": exp_dg})

    return pd.DataFrame(rows)


def load_experimental_data(path, ligand_column="ligand_id", dg_column="exp_dg"):
    if path is None:
        return pd.DataFrame(columns=["ligand_id", "exp_dg", "ligand_id_norm", "cinnabar_label"])

    path = Path(path)
    if not path.exists():
        print(f"Experimental data not found: {path}. Continuing without experimental comparison.")
        return pd.DataFrame(columns=["ligand_id", "exp_dg", "ligand_id_norm", "cinnabar_label"])

    if path.suffix.lower() in {".sdf", ".sd"}:
        exp_df = load_experimental_sdf(path)
    else:
        sep = "	" if path.suffix.lower() in {".tsv", ".tab"} else ","
        exp_df = pd.read_csv(path, sep=sep)
        if ligand_column not in exp_df.columns or dg_column not in exp_df.columns:
            raise ValueError(
                f"Experimental table must contain '{ligand_column}' and '{dg_column}' columns."
            )
        exp_df = exp_df.rename(columns={ligand_column: "ligand_id", dg_column: "exp_dg"})

    exp_df = exp_df.dropna(subset=["ligand_id", "exp_dg"]).copy()
    exp_df["exp_dg"] = pd.to_numeric(exp_df["exp_dg"], errors="raise")
    exp_df["ligand_id_norm"] = exp_df["ligand_id"].map(normalize_ligand_id)
    exp_df["cinnabar_label"] = exp_df["ligand_id"].map(cinnabar_label)
    return exp_df


exp_df = load_experimental_data(EXPERIMENTAL_DATA)
display(exp_df)


## Choose Analysis Columns

BATTER's merged `relative.csv` and `absolute.csv` tables include uncorrected
Cinnabar columns and, when available, SFC cycle-closure columns. Toggle
`USE_CYCLE_CLOSURE` in the configuration cell to choose which values drive
the comparisons below.


In [ ]:
requested_cycle_closure = USE_CYCLE_CLOSURE
has_relative_cycle_closure = {
    "DDG_cycle_closure",
    "uncertainty_cycle_closure",
}.issubset(relative_df.columns)
has_absolute_cycle_closure = {
    "DG_cycle_closure",
    "uncertainty_cycle_closure_path_dependent",
}.issubset(absolute_df.columns)

if requested_cycle_closure and not has_relative_cycle_closure:
    print("Cycle-closure relative columns are not present; using uncorrected values.")
    USE_CYCLE_CLOSURE = False

VALUE_MODE = "cycle_closure" if USE_CYCLE_CLOSURE else "uncorrected"
VALUE_MODE_LABEL = "cycle-closure corrected" if USE_CYCLE_CLOSURE else "uncorrected"
RELATIVE_VALUE_COLUMN = "DDG_cycle_closure" if USE_CYCLE_CLOSURE else "DDG"
RELATIVE_UNCERTAINTY_COLUMN = "uncertainty_cycle_closure" if USE_CYCLE_CLOSURE else "uncertainty"

if USE_CYCLE_CLOSURE and has_absolute_cycle_closure:
    ABSOLUTE_VALUE_COLUMN = "DG_cycle_closure"
    ABSOLUTE_UNCERTAINTY_COLUMN = "uncertainty_cycle_closure_path_dependent"
else:
    ABSOLUTE_VALUE_COLUMN = "DG"
    ABSOLUTE_UNCERTAINTY_COLUMN = "uncertainty"

missing_relative_columns = {
    RELATIVE_VALUE_COLUMN,
    RELATIVE_UNCERTAINTY_COLUMN,
} - set(relative_df.columns)
if missing_relative_columns:
    raise KeyError(f"Missing relative columns: {sorted(missing_relative_columns)}")

if not absolute_df.empty:
    missing_absolute_columns = {
        ABSOLUTE_VALUE_COLUMN,
        ABSOLUTE_UNCERTAINTY_COLUMN,
    } - set(absolute_df.columns)
    if missing_absolute_columns:
        raise KeyError(f"Missing absolute columns: {sorted(missing_absolute_columns)}")

print(f"Using {VALUE_MODE_LABEL} values for analysis.")


## Rebuild A Cinnabar Map For Plotting


In [ ]:
def filter_relative_edges(df):
    work = df.copy()
    work["labelA_norm"] = work["labelA"].map(normalize_ligand_id)
    work["labelB_norm"] = work["labelB"].map(normalize_ligand_id)

    if EXCLUDED_LIGANDS:
        excluded = {normalize_ligand_id(label) for label in EXCLUDED_LIGANDS}
        work = work.loc[
            ~work["labelA_norm"].isin(excluded)
            & ~work["labelB_norm"].isin(excluded)
        ].copy()

    return work


def mean_temperature():
    for column in ["temperature", "temperature_K"]:
        if column in raw_signed.columns:
            values = pd.to_numeric(raw_signed[column], errors="coerce").dropna()
            if not values.empty:
                return float(values.mean())
    return 298.15


def make_cinnabar_edges(df):
    work = df.dropna(subset=[RELATIVE_VALUE_COLUMN, RELATIVE_UNCERTAINTY_COLUMN]).copy()
    return pd.DataFrame({
        "edge_label": work["labelA"].astype(str) + "~" + work["labelB"].astype(str),
        "total_dG": pd.to_numeric(work[RELATIVE_VALUE_COLUMN], errors="raise"),
        "total_se": pd.to_numeric(work[RELATIVE_UNCERTAINTY_COLUMN], errors="raise"),
        "run_id": VALUE_MODE,
        "status": "success",
        "temperature": mean_temperature(),
    })


relative_for_analysis = filter_relative_edges(relative_df)
cinnabar_edges = make_cinnabar_edges(relative_for_analysis)

experimental_for_cinnabar = None
if not exp_df.empty:
    experimental_for_cinnabar = exp_df.rename(
        columns={"cinnabar_label": "ligand", "exp_dg": "abfe"}
    )[["ligand", "abfe"]].dropna().copy()

cinnabar_result = None
cinnabar_graph = None
try:
    cinnabar_result = dataframe_to_cinnabar(
        cinnabar_edges,
        ligand_column="edge_label",
        dg_column="total_dG",
        se_column="total_se",
        run_column="run_id",
        experimental_df=experimental_for_cinnabar,
        exp_ligand_column="ligand",
        exp_abfe_column="abfe",
        exp_source="experiment",
        merge_bidirectional=True,
    )
except RuntimeError as exc:
    print(
        "Skipping Cinnabar FEMap plots because `cinnabar` and/or `openff.units` "
        f"are not available in this kernel: {exc}"
    )

if cinnabar_result is not None:
    cinnabar_graph = cinnabar_result.femap.to_legacy_graph()
    print(
        f"Cinnabar graph built from {RELATIVE_VALUE_COLUMN} and "
        f"{RELATIVE_UNCERTAINTY_COLUMN}."
    )
    display(cinnabar_result.edge_summary)
    if cinnabar_result.exp_summary is not None:
        display(cinnabar_result.exp_summary)
else:
    display(cinnabar_edges)


## Relative Free-Energy Comparison


In [ ]:
relative = relative_for_analysis.copy()
relative["value_mode"] = VALUE_MODE
relative["sim_DDG"] = pd.to_numeric(relative[RELATIVE_VALUE_COLUMN], errors="raise")
relative["sim_uncertainty"] = pd.to_numeric(relative[RELATIVE_UNCERTAINTY_COLUMN], errors="raise")

if not exp_df.empty:
    exp_lookup = exp_df.set_index("ligand_id_norm")["exp_dg"]
    relative["exp_DDG"] = relative["labelB_norm"].map(exp_lookup) - relative["labelA_norm"].map(exp_lookup)
    relative["sim_minus_exp"] = relative["sim_DDG"] - relative["exp_DDG"]
    relative["abs_error"] = relative["sim_minus_exp"].abs()
else:
    relative["exp_DDG"] = np.nan
    relative["sim_minus_exp"] = np.nan
    relative["abs_error"] = np.nan

comparison_cols = [
    "labelA",
    "labelB",
    "value_mode",
    "sim_DDG",
    "sim_uncertainty",
    "exp_DDG",
    "sim_minus_exp",
    "abs_error",
]
for col in [
    "DDG",
    "uncertainty",
    "DDG_uncorrected",
    "uncertainty_uncorrected",
    "DDG_cycle_closure",
    "uncertainty_cycle_closure",
]:
    if col in relative.columns and col not in comparison_cols:
        comparison_cols.append(col)

relative_comparison = relative[comparison_cols].sort_values(
    ["abs_error", "labelA", "labelB"],
    ascending=[False, True, True],
    na_position="last",
)
display(relative_comparison)

relative_comparison.to_csv(
    OUTPUT_DIR / f"relative_experimental_comparison_{VALUE_MODE}.csv",
    index=False,
)


In [ ]:
plot_df = relative.dropna(subset=["sim_DDG", "exp_DDG"]).copy()

if plot_df.empty:
    print("No experimental DDG values are available for a scatter comparison.")
else:
    plot_df["ligand"] = plot_df["labelA"].astype(str) + "~" + plot_df["labelB"].astype(str)
    display(plot_df[["ligand", "value_mode", "sim_DDG", "sim_uncertainty", "exp_DDG"]])

    fig, ax = plt.subplots(figsize=(8, 8))
    if sns is not None:
        sns.scatterplot(data=plot_df, x="sim_DDG", y="exp_DDG", hue="ligand", ax=ax)
    else:
        for ligand, group in plot_df.groupby("ligand", sort=True):
            ax.scatter(group["sim_DDG"], group["exp_DDG"], label=ligand)
    ax.errorbar(
        data=plot_df,
        linestyle="",
        x="sim_DDG",
        y="exp_DDG",
        xerr="sim_uncertainty",
        ecolor="0.25",
        elinewidth=1,
        capsize=3,
    )

    annotation_offsets = [(6, 6), (6, -12), (-54, 6), (-54, -12)]
    for idx, row in enumerate(plot_df.itertuples(index=False)):
        ax.annotate(
            row.ligand,
            (row.sim_DDG, row.exp_DDG),
            xytext=annotation_offsets[idx % len(annotation_offsets)],
            textcoords="offset points",
            fontsize=8,
            alpha=0.9,
            bbox={"boxstyle": "round,pad=0.15", "fc": "white", "ec": "none", "alpha": 0.65},
        )

    ax.set_aspect("equal", adjustable="box")
    ax_min = min(ax.get_xlim()[0], ax.get_ylim()[0])
    ax_max = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.set_xlim(ax_min, ax_max)
    ax.set_ylim(ax_min, ax_max)

    x_linspace = np.linspace(*ax.get_xlim())
    ax.plot(x_linspace, x_linspace, ls="--", c=".3")
    ax.fill_between(x_linspace, x_linspace - 1, x_linspace + 1, color=".3", alpha=0.2)
    ax.fill_between(x_linspace, x_linspace - 2, x_linspace + 2, color=".3", alpha=0.1)

    ax.set_xlabel(f"BATTER RBFE DDG ({VALUE_MODE_LABEL}, kcal/mol)")
    ax.set_ylabel("Experimental RBFE DDG (kcal/mol)")
    ax.set_title(f"RBFE comparison ({VALUE_MODE_LABEL})")
    ax.legend(loc="best", fontsize=8, frameon=True)
    fig.tight_layout()

    scatter_plot = OUTPUT_DIR / f"rbfe_scatter_{VALUE_MODE}.png"
    fig.savefig(scatter_plot, dpi=300, bbox_inches="tight")
    plt.show()
    display(Image(filename=str(scatter_plot)))


In [ ]:
try:
    from cinnabar import plotting as cinnabar_plotting
except ImportError:
    cinnabar_plotting = None

if cinnabar_result is None:
    print("Cinnabar FEMap was not built; skipping Cinnabar DDG plot.")
elif cinnabar_plotting is None:
    print("Cinnabar plotting is not available in this kernel; skipping Cinnabar DDG plot.")
elif cinnabar_result.exp_summary is None:
    print("No experimental data were attached to the Cinnabar graph; skipping Cinnabar DDG plot.")
else:
    ddg_plot = OUTPUT_DIR / f"cinnabar_ddg_{VALUE_MODE}.png"
    cinnabar_plotting.plot_DDGs(
        cinnabar_graph,
        method_name="BATTER",
        target_name=f"RBFE ({VALUE_MODE_LABEL})",
        filename=str(ddg_plot),
    )
    display(Image(filename=str(ddg_plot)))


In [ ]:
metrics_df = relative.dropna(subset=["sim_DDG", "exp_DDG"]).copy()

if metrics_df.empty:
    print("No experimental values are available for error metrics.")
else:
    metrics = pd.DataFrame({
        "value_mode": [VALUE_MODE],
        "n_edges": [len(metrics_df)],
        "mean_signed_error": [metrics_df["sim_minus_exp"].mean()],
        "mean_abs_error": [metrics_df["sim_minus_exp"].abs().mean()],
        "rmse": [np.sqrt(np.mean(metrics_df["sim_minus_exp"] ** 2))],
        "pearson_r": [metrics_df[["sim_DDG", "exp_DDG"]].corr().iloc[0, 1]],
    })
    display(metrics)
    metrics.to_csv(OUTPUT_DIR / f"relative_metrics_{VALUE_MODE}.csv", index=False)


## Absolute Cinnabar Estimates


In [ ]:
if absolute_df.empty:
    print("No absolute Cinnabar estimates were found for this bundle.")
else:
    absolute = absolute_df.copy()
    absolute["label_norm"] = absolute["label"].map(normalize_ligand_id)
    if EXCLUDED_LIGANDS:
        excluded = {normalize_ligand_id(label) for label in EXCLUDED_LIGANDS}
        absolute = absolute.loc[~absolute["label_norm"].isin(excluded)].copy()

    absolute["value_mode"] = VALUE_MODE
    absolute["sim_DG"] = pd.to_numeric(absolute[ABSOLUTE_VALUE_COLUMN], errors="raise")
    absolute["sim_uncertainty"] = pd.to_numeric(absolute[ABSOLUTE_UNCERTAINTY_COLUMN], errors="raise")

    if not exp_df.empty:
        exp_lookup = exp_df.set_index("ligand_id_norm")["exp_dg"]
        absolute["exp_dg"] = absolute["label_norm"].map(exp_lookup)
        absolute["DG_minus_exp"] = absolute["sim_DG"] - absolute["exp_dg"]
        absolute["abs_error"] = absolute["DG_minus_exp"].abs()
    else:
        absolute["exp_dg"] = np.nan
        absolute["DG_minus_exp"] = np.nan
        absolute["abs_error"] = np.nan

    absolute_comparison_cols = [
        "label",
        "value_mode",
        "sim_DG",
        "sim_uncertainty",
        "exp_dg",
        "DG_minus_exp",
        "abs_error",
    ]
    for col in [
        "DG",
        "uncertainty",
        "DG_uncorrected",
        "uncertainty_uncorrected",
        "DG_cycle_closure",
        "uncertainty_cycle_closure_path_dependent",
        "uncertainty_cycle_closure_path_independent",
    ]:
        if col in absolute.columns and col not in absolute_comparison_cols:
            absolute_comparison_cols.append(col)

    absolute_comparison = absolute[absolute_comparison_cols].sort_values("sim_DG")
    display(absolute_comparison)
    absolute_comparison.to_csv(
        OUTPUT_DIR / f"absolute_experimental_comparison_{VALUE_MODE}.csv",
        index=False,
    )


In [ ]:
def plot_cinnabar_dgs(graph, filename):
    x_data = np.asarray([node[1]["exp_DG"] for node in graph.nodes(data=True)])
    y_data = np.asarray([node[1]["calc_DG"] for node in graph.nodes(data=True)])
    xerr = np.asarray([node[1]["exp_dDG"] for node in graph.nodes(data=True)])
    yerr = np.asarray([node[1]["calc_dDG"] for node in graph.nodes(data=True)])

    x_data = x_data - np.mean(x_data)
    y_data = y_data - np.mean(y_data)

    cinnabar_plotting._master_plot(
        x_data,
        y_data,
        xerr=xerr,
        yerr=yerr,
        origins=False,
        statistics=["RMSE", "MUE"],
        quantity=r"$\Delta$ G",
        method_name="BATTER",
        target_name=f"RBFE ({VALUE_MODE_LABEL})",
        filename=str(filename),
    )


if cinnabar_result is None:
    print("Cinnabar FEMap was not built; skipping Cinnabar DG plot.")
elif cinnabar_plotting is None:
    print("Cinnabar plotting is not available in this kernel; skipping Cinnabar DG plot.")
elif cinnabar_result.exp_summary is None:
    print("No experimental data were attached to the Cinnabar graph; skipping Cinnabar DG plot.")
else:
    dg_plot = OUTPUT_DIR / f"cinnabar_dg_{VALUE_MODE}.png"
    plot_cinnabar_dgs(cinnabar_graph, dg_plot)
    display(Image(filename=str(dg_plot)))


## Generated Bundle Assets


In [ ]:
for image_name in [
    "cinnabar_network.png",
    "cinnabar_dg_values.png",
    "cinnabar_absolute_sorted.png",
    "cinnabar_dg.png",
    "cinnabar_ddg.png",
    "cycle_closure_network.png",
    "cycle_closure_dg_values.png",
    "cycle_closure_errors.png",
]:
    image_path = CINNABAR_DIR / image_name
    if image_path.exists():
        print(image_name)
        display(Image(filename=str(image_path)))

dashboard = CINNABAR_DIR / "cinnabar_dashboard.html"
if dashboard.exists():
    display(HTML(f'<a href="{dashboard.as_posix()}" target="_blank">Open Cinnabar dashboard</a>'))
